Import libraries

In [13]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Tarnet/main_tarnet.ipynb")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [14]:
%time train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
%time test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

CPU times: user 23.9 ms, sys: 6.99 ms, total: 30.9 ms
Wall time: 30.2 ms
CPU times: user 9.7 ms, sys: 7.02 ms, total: 16.7 ms
Wall time: 16.6 ms
CPU times: user 4.63 ms, sys: 0 ns, total: 4.63 ms
Wall time: 4.6 ms


In [15]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [16]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [17]:
# Cách làm đúng: Chỉ tính trên khách hàng có phát sinh giao dịch
revenue_only = y_train[y_train > 0]
if len(revenue_only) > 0:
    threshold = np.percentile(revenue_only, 99) # Lấy top 1% của những người mua hàng
    y_train_clipped = np.clip(y_train, 0, threshold)
    y_val = np.clip(y_val, 0, threshold)
    print(f"Ngưỡng clip thực tế: {threshold}")


Ngưỡng clip thực tế: 499.0


In [18]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.78953321  1.6460815   1.41745802 -1.11692492  0.91314538  1.07643872
   0.9950542  -0.36894675 -0.88924565  1.13133938]]


In [19]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [20]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25615, 10]); y=torch.Size([25615, 1]); t=torch.Size([25615, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [21]:
epochs = 150
lr = 5e-5
wd = 1e-5
early_stop_metric = "ema_qini"
ema = True
ema_alpha = 0.35
patience = 20
shared_dropout = 0      
outcome_dropout = 0.
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")
print (f" activation = {activation}")

 epochs = 150
 learning rate = 5e-05
 weight decay = 1e-05
 early stop = ema_qini
 use ema = True
 ema alpha = 0.35
 patience = 20
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30
 activation = <class 'torch.nn.modules.activation.ReLU'>


In [22]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence CFRNet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            tarnet.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p= tarnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 0. Best value: 1.15005:   2%|▏         | 1/50 [01:14<1:00:33, 74.16s/it]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 1. Best value: 1.16307:   4%|▍         | 2/50 [02:30<1:00:16, 75.34s/it]

...

Best trial: 13. Best value: 1.18371:  96%|█████████▌| 48/50 [1:00:28<02:34, 77.18s/it]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 13. Best value: 1.18371:  98%|█████████▊| 49/50 [1:01:46<01:17, 77.38s/it]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 13. Best value: 1.18371: 100%|██████████| 50/50 [1:03:05<00:00, 75.72s/it]


In [23]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 13
Best mean_Val_AUQC: 1.183715
Best params: {'lr': 8.82782155013677e-05, 'weight_decay': 1.039251434722866e-05}

Best config table:


,lr,weight_decay,mean_Val_AUQC,std_Val_AUQC
0,0.000088,0.00001,1.183715,0.173838



Top 10 trials:


,trial,lr,weight_decay,mean_Val_AUQC,std_Val_AUQC
0,13,0.000088,0.000010,1.183715,0.173838
1,48,0.000106,0.000035,1.181011,0.176720
2,45,0.000120,0.000011,1.179696,0.189316
3,49,0.000101,0.000035,1.179539,0.169756
4,32,0.000098,0.000014,1.179522,0.173581
5,41,0.000099,0.000016,1.178908,0.177279
6,37,0.000120,0.000012,1.178658,0.187856
7,11,0.000119,0.000013,1.178520,0.196301
8,15,0.000091,0.000015,1.177866,0.179136
9,22,0.000092,0.000017,1.177782,0.177926


In [27]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])

print("Đang đánh giá trên test với best config từ validation:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}")
print(f"Số seed: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=shared_hidden,
        outcome_hidden=outcome_hidden,
        outcome_dropout=outcome_dropout,
        shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        activation=activation
    )

    tarnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'CHI TIẾT TỪNG SEED (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'KẾT QUẢ TRUNG BÌNH TEST (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Đang đánh giá trên test với best config từ validation:
  lr=8.8e-05, weight_decay=1.0e-05
Số seed: 5
Locked random seed: 412312
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: EMA_QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Best EMA Qini (alpha=0.35)
   Restore to epoch with highest smoothed (EMA) Qini score
   Patience: 20 epochs
Epoch 1/150 | Loss: 0.0103 | Val Loss: 193.7392 | Val Qini: 0.5491 | EMA Qini: 0.5491 | Best EMA: 0.5491 ⭐ NEW BEST EMA
Epoch 2/150 | Loss: 468.6473 | Val Loss: 193.6911 | Val Qini: 0.6399 | EMA Qini: 0.5809 | Best EMA: 0.5809 ⭐ NEW BEST EMA
Epoch 3/150 | Loss: 0.0143 | Val Loss: 193.6376 | Val Qini: 0.7091 | EMA Qini: 0.6258 | Best EMA: 0.6258 ⭐ NEW BEST EMA
Epoch 4/150 | Loss: 0.0179 | Val Loss: 193.5815 | Val Qini: 0.8204 | EMA Qini: 0.6939 | Best EMA: 0.6939 ⭐ NEW BEST EMA
Epoch 5/150 | Loss: 0.0231 | Val Loss: 193.5197 | Val Qini: 0.8629 | EMA Qini: 0.7530 | Best EMA: 0.7530 ⭐ NEW BEST EMA
Epoch 6/150 | Loss: 0.0322 | Val Loss: 193.4518 | Val Qini: 0.